# GSB솔루션 공정 품질 분석 — 전장길이 추가

**실행: Run → Run All Cells**. 실제 입력값으로 공정 품질을 분석합니다.

기본 입력은 **`data/raw/gsb_input_2.xlsx`**입니다. 파일이 없으면 기존 엑셀, 표준 CSV 순으로 사용합니다.
원본을 수정하지 않으며 결과는 실행별 폴더에 저장합니다. 신규 전장길이 규격은 성형·전조 모두 **79.30~80.70 mm**입니다.

LOT 기록값의 관계를 분석합니다. 개별 제품의 전후 동일시료인지, 평균/대표값인지는 확인이 필요합니다.

[사용 안내](docs/user-guide.md) · [입력·분석 기준](docs/data-guide.md)

## 1. 실행 준비
입력을 직접 지정하려면 `INPUT_FILE`을 파일 경로로 바꿉니다. 기존 커널에서도 수정된 분석 모듈을 다시 읽습니다.

In [ ]:
from pathlib import Path
import sys, importlib
from IPython.display import display, Markdown
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / 'src/gsb_analysis.py').exists():
    raise RuntimeError('프로젝트 루트에서 노트북을 실행하세요.')
sys.path.insert(0, str(ROOT))
import src.gsb_analysis as core
import src.gsb_extended_analysis as extended
importlib.reload(core)
importlib.reload(extended)
INPUT_FILE = core.select_input(ROOT)
print(f'분석 입력: {INPUT_FILE.name}')

## 2. 입력 검증과 원문 오류 확인
구형 O~R열과 신형 O~T열 헤더를 구분합니다. 결측은 0으로 바꾸지 않습니다.

새 파일의 `추가_25년!Q162`(LOT `250818-HD01-01-05`)에는 성형 전장길이가 **`79..8`**로 기록돼 있습니다.
`79.8`로 추정하지 않고, 아래에서 `length_errors='coerce'`를 명시해 **해당 길이만 결측 처리**합니다.
이 LOT의 유효한 외경·불량수량은 해당 분석에 사용합니다. 원문과 위치는 오류 표 및 CSV에 보존됩니다.
엄격하게 오류 시 중단하려면 `length_errors='raise'`를 사용하세요.

In [ ]:
data = core.load_data(INPUT_FILE, length_errors='coerce')
if set(data['part_no']) != {'11404 10806P'}:
    raise ValueError('현재 규격은 11404 10806P용입니다. 다른 품번의 규격은 별도 확인하세요.')
tables = core.summarize(data)
tables.update(extended.summarize_extended(data))
tables['quality_checks'] = core.quality_checks(data)
print(f'자료 로딩 완료: {len(data):,}개 LOT')
display(data.head(10))
display(tables['length_input_issues'])
display(tables['quality_checks'])

## 3. 치수 확보율과 규격 비교
분석 쌍마다 필요한 값이 모두 유효한 LOT만 사용합니다. 네 치수 전체가 유효한 LOT 수와 두 외경만 연결되는 LOT 수는 다릅니다.
유효 기록이 없는 항목의 이탈 수 0은 정상 판정 근거가 아닙니다.

In [ ]:
display(tables['coverage_extended'].round(2))
specifications = tables['specifications'].copy()
specifications['항목'] = specifications['항목'].replace(extended.LABELS)
display(specifications)
display(tables['defect_counts'].round(2))
print('PPM은 생산수량·불량수량이 함께 유효한 LOT 기준의 잠정 지표입니다.')
print('규격 내 기록값이 LOT 전체 무불량을 의미하지 않습니다.')

## 4. 연도별 성형 설비 구성
엑셀 LOT 행에 기록된 실제 성형 설비를 집계합니다. 설비 전환과 연도 효과가 겹칠 수 있으므로 설비 자체의 인과효과로 해석하지 않습니다.

In [ ]:
figures = core.plots(data)
figures.update(extended.extended_plots(data, tables))
plt.close('all')  # 표시할 단계에서만 display하여 자동 중복 출력을 방지
display(tables['equipment_composition'].round(2))
display(figures['equipment_composition'])

## 5. 전조하경과 나사외경의 관계
동일 LOT의 두 외경 기록을 사용하며, 원 면적은 동일 좌표의 LOT 수에 비례합니다. 측정값에 임의의 흔들림을 넣지 않습니다.
전체 상관계수가 작더라도 연도별 관계나 다른 공정변수의 영향을 배제할 수 없습니다.

In [ ]:
display(tables['correlation'].round(4))
display(figures['diameter_by_year'])
display(figures['histograms'])
display(figures['trend'])

## 6. 성형·전조 전장길이의 관계
두 길이가 같은 LOT에서 함께 커지거나 작아지는지를 봅니다. 전체와 연도별 피어슨 상관계수를 함께 표시합니다.
전체 상관계수에는 연도별 평균 이동이 포함될 수 있으며, 동일 제품의 전후 측정이라는 의미는 아닙니다.

In [ ]:
display(tables['length_correlation'].round(4))
display(figures['length_scatter'])
display(figures['length_correlation'])
display(figures['length_histograms'])

## 7. 연도별 평균과 변화량
각 치수의 유효 기록 전체를 사용해 평균을 계산합니다(회귀 모델의 공통 표본과 다릅니다).
2024~2026년의 세 시점만으로 지속적인 증가·감소 추세나 공정 변경의 인과효과를 확정하지 않습니다.

In [ ]:
display(tables['dimension_statistics'].round(6))
display(tables['diameter_mean_changes'].round(6))
display(figures['diameter_year_means'])

## 8. 전조 불량의 생산시기·LOT 집중도
월은 **LOT 코드 앞 6자리의 생산일자** 기준입니다. 실제 전조 검사월 및 시트 연도와 다를 수 있습니다.
상위 LOT 비중의 분모는 결측을 제외한 전체 기록 전조 불량수량입니다. Pareto 그래프는 상위 20개만 표시하지만 누적 비중의 분모는 전체입니다.

In [ ]:
display(tables['monthly_thread_defects'])
display(tables['defect_concentration'].round(2))
display(tables['top_defect_lots'].head(20).round(2))
for name in ['monthly_thread_defects', 'defect_pareto']:
    if name in figures:
        display(figures[name])

## 9. 시간순 나사외경 회귀 평가
절편을 포함한 선형 최소제곱 회귀(OLS)를 사용합니다. 입력은 **성형 전조하경**, 또는 **성형 전조하경+성형 전장길이**이며, 목표는 전조 나사외경입니다.
전조 후 전장길이는 예측 입력으로 사용하지 않습니다. Baseline은 **학습 자료의 나사외경 평균**입니다.

- 2025년 평가: 2024년 학습 → 2025년 평가
- 2026년 평가: 2024~2025년 학습 → 2026년 평가

이는 **시트 연도 기준의 전진 검증**입니다. 시트 연도와 실제 측정연도가 다른 LOT가 있어 실제 검사시간 기준의 엄밀한 검증과는 구분해야 합니다.

먼저 모델별 유효 표본으로 계산합니다. 요약 표의 Baseline은 1변수 모델 표본 기준입니다.
2025년 두 모델의 평가 표본은 56/55 LOT이고, 2026년 학습 표본은 161/160 LOT로 다릅니다.
아래에는 각 모델 자신의 Baseline 및 두 모델의 **학습·평가 LOT를 동일하게 맞춘 비교**도 제공합니다.
이 검증은 연속 치수 예측이며 **불량확률 예측이 아닙니다**.

In [ ]:
tables.update(extended.temporal_regression(data))
print('회귀 평가 요약 — MAE 단위: mm, 작을수록 오차가 적음')
display(tables['regression_summary'].round(5))
print('모델별 표본수와 각 모델의 학습 평균 Baseline')
display(tables['regression_metrics'].loc[tables['regression_metrics']['평가 기준'].str.startswith('모델별')].round(6))
print('동일 학습·평가 LOT를 사용한 비교 — MAE 개선량이 양수이면 Baseline보다 개선')
display(tables['regression_metrics'].loc[tables['regression_metrics']['평가 기준'].eq('동일 LOT 비교')].round(6))
display(tables['regression_predictions'].head(10).round(6))

## 10. 기록값 점검과 해석 한계
이미 기록된 네 치수의 규격을 점검합니다. 미래 불량 사전알람이 아닙니다.
불량품 실측값, 불량유형·판정기준, LOT 값의 대표성, 설비·금형·설정값 변경 이력이 추가로 필요합니다.

In [ ]:
tables['measured_flags'] = core.measured_flags(data)
flags = tables['measured_flags']
mask = flags.filter(like='_status').ne('기록값 규격 내').any(axis=1)
display(flags.loc[mask].head(30))
if tables['specifications']['규격 이탈 수'].sum() == 0:
    print('유효 기록값에서 규격 이탈이 관측되지 않았습니다. 이것이 전체 생산품 무불량을 뜻하지는 않습니다.')
print('수량 불량과 치수 이탈을 연결할 불량유형·실측 라벨이 필요합니다.')
print('회귀 성능은 소수 연도에 대한 탐색 결과이며 일관된 개선 또는 실무 적용 성능으로 확정하지 않습니다.')

## 11. 결과 저장
모든 표(전체 LOT·개별 회귀 예측 포함), 그래프 및 HTML 보고서를 저장합니다.
입력 SHA-256, 규격, 길이 오류 수, 회귀 기준, 분석 코드 해시도 기록합니다.

In [ ]:
result_dir = core.save_results(data, tables, figures, INPUT_FILE, ROOT)
plt.close('all')
relative = result_dir.relative_to(ROOT).as_posix()
display(Markdown(f'## 분석 완료\n[결과 폴더 열기]({relative}/) · [HTML 보고서]({relative}/report.html)'))
print(f'저장 위치: {relative}')